# End-to-end example: Ce and Gd temperature study

A full pass over a small study: load the files, check the acquisition settings, describe the study explicitly, pick emission lines from the NIST Handbook, integrate them, and compare each temperature against 300 °C.

The data folder `data/end_to_end` holds Cerium and Gadolinium LIBS spectra in LiCl-KCl at 1 and 5 wt.%, each measured at 300, 350 and 400 °C:

```
data/end_to_end/
├── Ce/1_wt%/   Ce/5_wt%/
└── Gd/1_wt%/   Gd/5_wt%/
```

Requirements: the package installed in this kernel (`pip install -e .` from the repo root) and a NIST database (`libs-viewer nist build`, see step 5).

> **Just want the results?** `LV.run_study(...)` does all of this in one call, including spotting the mixed-settings file. See [quick_study.ipynb](quick_study.ipynb). This notebook shows what happens inside it.

## 1. Load the data

First step as always is to load the data. `LoadLIBSData` reads every `.tdms` file under the folder (`recursive=True` includes the sub-folders) and behaves like a read-only dict keyed by the path relative to that folder.

`overview()` puts the settings that matter for a fair comparison side by side.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import libs_viewer.libs as LV
from libs_viewer.libs.plotting import plot_spectrum
from libs_viewer.nist import query

# The repository root, found from wherever the notebook is started (repo root or examples/)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())

data = LV.LoadLIBSData(ROOT / 'data' / 'end_to_end', recursive=True)
data.overview()

## 2. Check the acquisition settings

The loader warned that one file has **parameters that differ between grid locations**. `overview()` lists them in the *mixed settings* column, and `varying_params()` shows the values for every location:

In [ ]:
odd = data['Gd/5_wt%/Gd_Sample5_300_3.5delay.tdms']
odd.varying_params()

Only `Grid Location # 1-1` was acquired like the rest of the Gd 5 wt% series (3.5 µs gate, 80 mJ, 5 accumulated shots). Locations `2-1 … 2-10` are a different acquisition (1.0 µs gate, 10 mJ), so they must be left out. The manifest below does that.

Some differences are expected and harmless: each sample keeps its own gate delay (Ce 1 wt% 1.0 µs, Gd 1 wt% 2.5 µs, the 5 wt% samples 3.5 µs), and samples are only ever compared with themselves at a different temperature. Accumulated and single-shot files are also fine, because every spectrum is put on a *counts per shot* scale first.

## 3. Describe the study (the manifest)

The file names don't follow one pattern (`Ce1400.tdms`, `Ce1 350 new location.tdms`, `Gd_Sample1_300.tdms`), so instead of guessing, every file is mapped explicitly to its element, sample and temperature. `use_columns` picks the grid locations to keep (`None` = all).

The same table, saved as a CSV, can be passed straight to `LV.study_plan` / `LV.run_study` as `manifest=` (`use_columns` as names separated by `;`). For folders whose names do follow a pattern, `pattern='{element}/{sample}/*{temperature}*'` builds it for you.

In [ ]:
manifest = pd.DataFrame([
    # file                                     element  sample      temperature  use_columns
    ('Ce/1_wt%/Ce1_300C.tdms',                 'Ce',    'Ce 1 wt%', 300,         None),
    ('Ce/1_wt%/Ce1 350 new location.tdms',     'Ce',    'Ce 1 wt%', 350,         None),
    ('Ce/1_wt%/Ce1400.tdms',                   'Ce',    'Ce 1 wt%', 400,         None),
    ('Ce/5_wt%/Ce_Sample2_Temp_300C.tdms',     'Ce',    'Ce 5 wt%', 300,         None),
    ('Ce/5_wt%/Ce_Sample2_Temp_350C.tdms',     'Ce',    'Ce 5 wt%', 350,         None),
    ('Ce/5_wt%/Ce_Sample2_Temp_400C.tdms',     'Ce',    'Ce 5 wt%', 400,         None),
    ('Gd/1_wt%/Gd_Sample1_300.tdms',           'Gd',    'Gd 1 wt%', 300,         None),
    ('Gd/1_wt%/Gd_Sample1_350.tdms',           'Gd',    'Gd 1 wt%', 350,         None),
    ('Gd/1_wt%/Gd_Sample1_400.tdms',           'Gd',    'Gd 1 wt%', 400,         None),
    ('Gd/5_wt%/Gd_Sample5_300_3.5delay.tdms',  'Gd',    'Gd 5 wt%', 300,         ['Grid Location # 1-1']),
    ('Gd/5_wt%/Gd_Sample5_350_3.5delay.tdms',  'Gd',    'Gd 5 wt%', 350,         None),
    ('Gd/5_wt%/Gd_Sample5_400_3.5delay.tdms',  'Gd',    'Gd 5 wt%', 400,         None),
], columns=['file', 'element', 'sample', 'temperature', 'use_columns'])

# Every loaded file must be in the manifest and vice versa
assert set(manifest.file) == set(data.keys()), set(manifest.file) ^ set(data.keys())
SAMPLES = list(manifest['sample'].unique())
REF_T = 300   # reference temperature for the ratios
manifest

Within each sample, the settings should match across temperatures. For the odd Gd file this checks the kept column, not just the first one:

In [ ]:
def settings(row):
    f = data[row.file]
    kept = f.column_params if row.use_columns is None else f.column_params[row.use_columns]
    p = kept.loc[['Gate Delay (usec)', 'Energy']]
    assert (p.nunique(axis=1) == 1).all(), f'{row.file}: kept columns still differ'
    return p.iloc[:, 0]

manifest.apply(settings, axis=1).set_index([manifest['sample'], manifest.temperature])

## 4. Per-shot spectra and saturated pixels

For every sample and temperature, `per_shot_spectrum` divides accumulated columns by the number of shots and averages the kept grid locations.

`saturated_mask` flags pixels where the detector clipped (~65 000 counts per shot) at *any* temperature of a sample. The same pixels are then dropped at every temperature for the *Total (excl. sat.)* row, so the totals stay comparable.

In [ ]:
wl = data[0].wavelengths           # all files share one wavelength axis
assert all(np.array_equal(f.wavelengths, wl) for f in data.data.values())

spectra, masks = {}, {}            # spectra[sample][temperature] -> per-shot spectrum
for sample, g in manifest.groupby('sample', sort=False):
    spectra[sample] = {t: LV.per_shot_spectrum(data[k], cols)
                       for k, t, cols in zip(g.file, g.temperature, g.use_columns)}
    masks[sample] = LV.saturated_mask([data[k] for k in g.file], list(g.use_columns))

pd.Series({s: int(m.sum()) for s, m in masks.items()}, name='saturated pixels')

## 5. Pick lines from the NIST Handbook

The line lookups need the local NIST database. Its default location is the per-user data folder (`libs-viewer paths`); build it once with `libs-viewer nist build`. If you still have the older copy in the repository root, the cell below falls back to it.

In [ ]:
NIST_DB = None                                  # None = the per-user database
if not query.database_info()['exists'] and (ROOT / 'nist_handbook.sqlite').exists():
    NIST_DB = ROOT / 'nist_handbook.sqlite'     # older copy in the repo root
info = query.database_info(NIST_DB)
info['path'], info['elements']

For each element, `select_lines` takes the strongest NIST lines and keeps those that show a clean peak in the 300 °C spectrum of its samples. It rejects lines that sit near a saturated pixel or within 0.15 nm of a strong line of the *other* element (Ce and Gd lines interleave in this region).

Two things specific to this data:

- **Calibration offset.** Peaks sit about 0.05 nm below the NIST wavelengths (Ce II 418.66 nm shows up at 418.60 nm). That is inside the default `peak_tol` of 0.1 nm, so nothing needs shifting.
- **Weak lines at 1 wt%.** Ce 1 wt% was recorded with a 1.0 µs gate, so its continuum is high and its lines are small on top of it: the prominence of Ce II 418.66 is ~300 counts on a ~900 count continuum. The default threshold (prominence ≥ 2 × the spectrum's median) finds no Ce line there at all, so `min_prominence=0.25` is used.

In [ ]:
ELEMENTS = {'Ce': 'Gd', 'Gd': 'Ce'}             # element -> the element that may interfere
line_tables = {}
for el, other in ELEMENTS.items():
    samples = manifest.loc[manifest.element == el, 'sample'].unique()
    line_tables[el] = LV.select_lines(
        el, wl,
        y_refs={s: spectra[s][REF_T] for s in samples},
        n=4, interferents=[other],
        exclude=np.logical_or.reduce([masks[s] for s in samples]),
        min_prominence=0.25, db_path=NIST_DB)

show = ['label', 'intensity', 'status', 'detected_in']
line_tables['Ce'][show].head(20)

In [ ]:
line_tables['Gd'][show].head(12)

Lines found in more samples are preferred, but a line is only integrated in the samples where it was actually detected:

In [ ]:
LINES = {}                                      # LINES[sample] = {label: wavelength_nm}
for sample, el in manifest.groupby('sample', sort=False).element.first().items():
    picked = line_tables[el].query("status == 'picked'")
    LINES[sample] = {r.label: r.wavelength_nm for r in picked.itertuples()
                     if sample in r.detected_in.split(', ')}
LINES

## 6. Look at the spectra

`plot_spectrum` returns a matplotlib `Figure` (it never calls `plt.show()`), so it displays when it is the last expression of a cell and can be saved with `fig.savefig(...)`. The picked lines are marked as reference lines.

For Ce 1 wt% the plot warns that it finds no peak at the two Ce lines. Its own peak finder uses the stricter default threshold, and the zoom shows why: at 300 °C both lines are only just above the noise.

In [ ]:
plot_spectrum(wl, spectra['Ce 5 wt%'][REF_T], title=f'Ce 5 wt%, {REF_T} °C',
              ref_lines=LINES['Ce 5 wt%'], wl_range=(395, 425))

In [ ]:
plot_spectrum(wl, spectra['Ce 1 wt%'][REF_T], title=f'Ce 1 wt%, {REF_T} °C',
              ref_lines=LINES['Ce 1 wt%'], wl_range=(395, 425))

In [ ]:
plot_spectrum(wl, spectra['Gd 5 wt%'][REF_T], title=f'Gd 5 wt%, {REF_T} °C',
              ref_lines=LINES['Gd 5 wt%'], wl_range=(330, 370))

## 7. Line areas and ratios

`analyze_sample` finds each line's integration window once, on the 300 °C spectrum, and uses the same pixels at every temperature. The line area is measured above a straight local baseline. The totals are the integral of the continuum-corrected (SNIP) spectrum, with and without the saturated pixels.

`ratio_tables` then gives, relative to 300 °C:

- `R_direct` = A(T) / A(300) for every row, so the *Total* rows are R_total, the change in overall emitted light.
- `R_rel` = [A_line(T) / A_total(T)] / [A_line(300) / A_total(300)], the change in the line *relative to the whole spectrum*, which cancels plasma-to-plasma changes in overall brightness.

In [ ]:
areas, ratios = {}, {}
for sample in SAMPLES:
    areas[sample] = LV.analyze_sample(wl, spectra[sample], LINES[sample],
                                      exclude=masks[sample], ref=REF_T)
    ratios[sample] = LV.ratio_tables(areas[sample], ref=REF_T)

pd.concat(areas, names=['sample', 'row']).round(1)

In [ ]:
pd.concat({s: r['R_rel'] for s, r in ratios.items()}, names=['sample', 'row']).round(3)

In [ ]:
fig, axes = plt.subplots(1, len(SAMPLES), figsize=(16, 4), sharey=True, layout='constrained')
for ax, sample in zip(axes, SAMPLES):
    r = ratios[sample]
    r['R_rel'].drop(index=[LV.TOTAL, LV.TOTAL_EXCL]).T.plot(ax=ax, marker='o')
    r['R_direct'].loc[LV.TOTAL].plot(ax=ax, color='k', ls='--', marker='s', label='R_total')
    ax.axhline(1, color='grey', lw=0.5)
    ax.set_title(sample)
    ax.set_xlabel('Temperature (°C)')
    ax.set_xticks(sorted(spectra[sample]))
    ax.legend(fontsize=8)
axes[0].set_ylabel(f'R_rel (lines), R_total (dashed), vs {REF_T} °C')
axes[0].set_yscale('log');

## 8. Location-to-location scatter

The ratios above use the average over grid locations. `per_column_areas` repeats the measurement for every grid location separately, with the same windows. The coefficient of variation (CV) between locations shows how far a ratio has to move before it means anything. It is listed for every file with more than one kept location:

In [ ]:
scatter = {}
for row in manifest.itertuples():
    f = data[row.file]
    kept = list(f.spectra.columns) if row.use_columns is None else row.use_columns
    if len(kept) < 2:
        continue
    windows = LV.line_windows(wl, spectra[row.sample][REF_T], LINES[row.sample])
    per_loc = LV.per_column_areas(f, LINES[row.sample], windows, columns=row.use_columns,
                                  exclude=masks[row.sample])
    cv = 100 * per_loc.std(axis=1) / per_loc.mean(axis=1)
    scatter[(row.sample, row.temperature)] = {'locations': len(kept),
                                              'line CV (%), mean over lines': cv[list(LINES[row.sample])].mean(),
                                              'Total CV (%)': cv[LV.TOTAL]}
pd.DataFrame(scatter).T.round(1)

**Reading the results**

- **Single locations scatter a lot** in every file: line CV of 50 to 270 %, total CV of 20 to 85 %. The mean over n locations is uncertain by roughly CV/√n, so about ±45 % for 10 locations at a 150 % CV. Files with a single location (Ce 5 wt% at 300/350 °C, Gd 1 wt% at 350/400 °C) give no scatter estimate at all.
- **Ce 1 wt%**: the lines are barely above the noise, so its ratios (0.86 to 1.57) cannot be told apart from 1.
- **Ce 5 wt%, Gd 1 wt%, Gd 5 wt%**: all lines of an element move together. That fits a real change, but some of it is plasma brightness that every line shares. Weigh it against the scatter above before drawing conclusions.
- **Gd 1 wt%** drops by about 10× from 300 to 350 °C, and its total light drops too. The 300 °C file has two locations and the others one, so check those spectra first.

## Next steps

- Run the whole study in one call: [quick_study.ipynb](quick_study.ipynb).
- `result.save(...)` writes the tables together with the settings that produced them (`settings.json`).
- Compare baseline methods: `plotting.plot_baseline(wl, y, method='airpls')`, or `method='airpls'` in `analyze_sample` (roadmap phase 4 benchmarks this properly).